# Lab: Train Your Model with Keras
## Purpose:
- Learn to train any Keras model, independent of its architecture.
- The role of the optimizer and loss function
 - Combining optimizer and loss to train a model.

### Topics:
- Keras
- Optimizer
- Loss

### Steps
* Load the dataset.
* Define a two-layer neural network model using Keras.
* Train the model using Keras implementations of the loss function, the Adam optimizer, and the training loop.

Date: 2026-02-25

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_8_train_your_model_with_keras.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
# imports
import os # For adjusting Keras settings.
os.environ['KERAS_BACKEND'] = 'jax' # Set a parameter for Keras.

import jax.numpy as jnp # For defining and working with vectors and matrices.
import keras # For defining and training neural nework models.
import pandas as pd # For displaying and loading data.

In [ ]:
# load data
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/food-water-dataset.csv")

# Extract embeddings and labels.
X_train = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
labels = df["Label"].values # Labels: "food" or "water".
# Convert labels to numeric values for training the model (food = 1, water = 0).
y_train = jnp.where(labels == "food", 1, 0)
df["Numeric label"] = y_train

# Print the loaded data for verification.
df.head(n=20)

## Define the neural network model

Implement `build_neural_network()` for defining a two-layer neural network using Keras.

The operations of the hidden layer are defined in
```python
operations.append(keras.layers.Dense(hidden_dim, activation="relu"))
```

The operations of the output layer are defined in
```python
operations.append(keras.layers.Dense(1, activation="sigmoid"))
```

Reflect upon why this model is using a sigmoid activation function as the output layer? Could this be replaced with a SoftMax? When would this be useful?

    Of course it could. Sigmoid is used for binary classification. Softmax is used for multiple-choice classification. Binary classification is a special case of multiple-choice classification.

In [ ]:
def build_neural_network(hidden_dim: int = 10) -> keras.Model:
  """
  Intializes a two-layer neural network for binary classification, implemented in Keras.
  The hidden layer uses a ReLU activation function.
  The ouput layer uses a sigmoid activation function.

  Args:
    hidden_dim: The dimension of the hidden layer.

  Returns:
    A keras.Model instance that implements the logistic regression model.
  """

  operations = []

  # Add the operations for a hidden layer with a ReLU activation function.
  operations.append(keras.layers.Dense(hidden_dim, activation="relu"))

  # Add the operations for an output layer with a sigmoid activation function.
  operations.append(keras.layers.Dense(1, activation="sigmoid"))

  # Construct a model such that inputs are passed sequentially through every
  # layer.
  model = keras.Sequential(operations)
  return model

------
>The ingredients for training a model**
>
>* **Loss function**: Function of the current model weights and predictions, and the target labels in training data. This is the function that we optimize during training. A lower value of this function means that the model is better at making predictions on examples in the training data.
>*  **Optimizer**: Updates the parameters (weights) of the model so the loss decreases. Updating is based on the gradient of the loss function wrt training examples.
>* All optimizers have to be initialized with the `learning_rate` parameter. This parameter defines how big each update step should be.
>* **Model**: Defines which computations are needed to process the input. When a model is defined using a deep learning framework such as Keras, this automatically defines the parameters for each layer. For example, when you define a layer using `Dense`, this also initializes the weights and the bias term for that layer.
>
------

## Define a loss function

Define both the loss function and the optimizer. Use existing implementations for both of these components.
------
> Define the loss function. Use binary cross-entropy loss. In Keras, loss functions are defined in the `keras.losses` module and the binary cross-entropy loss can be initialized using the following class:
>
```python
keras.losses.BinaryCrossentropy()
```

In [ ]:
# Define the optimizer.
optimizer = keras.losses.BinaryCrossentropy()

## Define the optimizer

Almost always use the Adam optimizer because it implements a more sophisticated version of the gradient update step than the regular SGD algorithm by adapting step size depending on the slope of the loss function.

------
> Define the optimizer using a learning rate of 0.01.
> The optimizer will compute the gradients and use them to update the parameters on each batch.
```python
keras.optimizers.Adam(learning_rate=<LEARNING RATE>)
```
> Weight decay strength can be set by adding the optional `weight_decay` parameter.
------

In [ ]:
# Define the optimizer.
optimizer = keras.optimizers.Adam(learning_rate=0.01)

## The `compile` method

Put loss and optimizer functions together for training.

#### Define the model
- An instance of the `keras.Model` class.
- Use `build_neural_network()` function from above to define an MLP.
- Use `compile()` to attach the loss function and the optimizer to the model.
- Can specify optional metrics, such as the accuracy. The result of applying the metric after each epoch will be printed as part of the training log.

In [ ]:
# Set a random seed for reproducibility.
keras.utils.set_random_seed(126)

# Define a model.
model = build_neural_network(hidden_dim = 10)

# Attach the loss function, the optimizer and metrics.
model.compile(loss=loss_fn, optimizer=optimizer, metrics=["accuracy"])

## Train the model

------
> use `model.fit()` with these arguments:
>* `x`: The input of the training data (a JAX array, `X_train` in this case).
>* `y`: the target values in the training data (a JAX array, `y_train` is this case).
>* `epochs`: The number of epochs. This specifies how many times the model loops through all training examples.
>* `batch_size` (optional): This specifies how many examples there should be in one mini-batch.
>* `validation_data` (optional): A tuple `(X_val, y_val)` to compute the validation loss and accuracy after each epoch.
>* `callbacks` (optional): A list of functions that should be executed at the end of each epoch. One useful function to include in this list is `keras.callbacks.EarlyStopping()` to implement early stopping.
------

In [ ]:
# Use the model.fit() method to train your model.
history = model.fit(x=X_train,
                    y=y_train,
                    epochs=10,
                    batch_size=8)